# SmartBite PP-OCRv5 Products Date Detection Balanced Fine-Tuning

Fine-tunes PP-OCRv5 server detection on the balanced Products date-only detection dataset. The dataset zip is created locally by `app.scripts.build_products_date_detection_dataset`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/dataset /content/workdir /content/output /content/PaddleOCR /content/wheels
!mkdir -p /content/dataset /content/workdir /content/output

In [ ]:
from pathlib import Path

DATASET_ZIP = Path('/content/drive/My Drive/sb-colab/products_date_detection_balanced.zip')
PADDELOCR_ZIP = Path('/content/drive/My Drive/sb-colab/PaddleOCR.zip')
BASE_UNZIP_DIR = Path('/content/dataset')
OUTPUT_DIR = Path('/content/output/ppocrv5_products_date_det_balanced')
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/ppocrv5_products_date_det_balanced')

EPOCHS = 40
BATCH_SIZE = 4
LEARNING_RATE = 0.0005

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
print('DATASET_ZIP =', DATASET_ZIP)

In [ ]:
import os
import shlex
import subprocess
import sys

def run_live(cmd, env=None, cwd=None):
    cmd = [str(x) for x in cmd]
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
run_live(['unzip', '-q', '-o', DATASET_ZIP, '-d', BASE_UNZIP_DIR])

def find_dataset_root(base: Path) -> Path:
    candidates = [p.parent for p in base.rglob('train_det_label.txt') if (p.parent / 'val_det_label.txt').exists()]
    assert candidates, 'Could not find dataset root containing train_det_label.txt and val_det_label.txt'
    return sorted(candidates, key=lambda p: len(str(p)))[0]

DATASET_ROOT = find_dataset_root(BASE_UNZIP_DIR)
train_label_file = DATASET_ROOT / 'train_det_label.txt'
val_label_file = DATASET_ROOT / 'val_det_label.txt'
test_label_file = DATASET_ROOT / 'test_det_label.txt'
for p in [train_label_file, val_label_file, test_label_file, DATASET_ROOT / 'summary.json']:
    print(p, p.exists())
    assert p.exists(), p

## Install PaddleOCR

In [ ]:
import shutil

if PADDELOCR_ZIP.exists():
    run_live(['unzip', '-q', '-o', PADDELOCR_ZIP, '-d', '/content'])
    candidates = [p for p in Path('/content').glob('PaddleOCR*') if (p / 'tools' / 'train.py').exists()]
    assert candidates, 'PaddleOCR.zip did not extract to a PaddleOCR repo'
    extracted = sorted(candidates, key=lambda p: len(str(p)))[0]
    if extracted != Path('/content/PaddleOCR'):
        if Path('/content/PaddleOCR').exists():
            shutil.rmtree('/content/PaddleOCR')
        extracted.rename('/content/PaddleOCR')
else:
    run_live(['git', 'clone', 'https://github.com/PaddlePaddle/PaddleOCR.git', '/content/PaddleOCR'])

has_gpu = os.system('nvidia-smi > /dev/null 2>&1') == 0
wheel_zip = Path('/content/drive/My Drive/colab-cu118.zip' if has_gpu else '/content/drive/My Drive/colab-cpu.zip')
if wheel_zip.exists():
    run_live(['unzip', '-q', '-o', wheel_zip, '-d', '/content/wheels'])
    whls = sorted(str(p) for p in Path('/content/wheels').rglob('*.whl'))
    if whls:
        run_live([sys.executable, '-m', 'pip', 'install', '-q', *whls])
else:
    print('Wheelhouse zip not found; installing PaddleOCR requirements from internet fallback.')
    run_live([sys.executable, '-m', 'pip', 'install', '-q', 'paddleocr'])

run_live([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/PaddleOCR/requirements.txt'])

## Training Config

In [ ]:
BASE_CONFIG = Path('/content/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_server_det.yml')
PRETRAINED_MODEL_CANDIDATES = [
    Path('/content/drive/My Drive/sb-colab/models/PP-OCRv5_server_det_pretrained.pdparams'),
    Path('/content/drive/My Drive/models/PP-OCRv5_server_det_pretrained.pdparams'),
]
PRETRAINED_MODEL = next((p for p in PRETRAINED_MODEL_CANDIDATES if p.exists()), None)
assert BASE_CONFIG.exists(), f'Missing base config: {BASE_CONFIG}'
assert PRETRAINED_MODEL is not None, 'Missing PP-OCRv5_server_det_pretrained.pdparams in Drive'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('BASE_CONFIG =', BASE_CONFIG)
print('PRETRAINED_MODEL =', PRETRAINED_MODEL)

## Train Detection

In [ ]:
cmd = [
    sys.executable, '/content/PaddleOCR/tools/train.py',
    '-c', str(BASE_CONFIG),
    '-o',
    f'Global.pretrained_model={PRETRAINED_MODEL}',
    f'Global.save_model_dir={OUTPUT_DIR}',
    f'Global.epoch_num={EPOCHS}',
    'Global.print_batch_step=10',
    'Global.eval_batch_step=[0,500]',
    'Global.save_epoch_step=1',
    f'Optimizer.lr.learning_rate={LEARNING_RATE}',
    f'Train.loader.batch_size_per_card={BATCH_SIZE}',
    'Eval.loader.batch_size_per_card=1',
    f'Train.dataset.data_dir={DATASET_ROOT}',
    f'Eval.dataset.data_dir={DATASET_ROOT}',
    f'Train.dataset.label_file_list=["{train_label_file}"]',
    f'Eval.dataset.label_file_list=["{val_label_file}"]',
]
if os.system('nvidia-smi > /dev/null 2>&1') != 0:
    cmd.append('Global.use_gpu=False')

env = dict(os.environ)
env['PYTHONPATH'] = '/content/PaddleOCR'
env['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
run_live(cmd, env=env)
print('Detection training finished:', OUTPUT_DIR)

## Final Test Evaluation

In [ ]:
BEST_BASE = OUTPUT_DIR / 'best_accuracy'
BEST_PARAMS = Path(str(BEST_BASE) + '.pdparams')
assert BEST_PARAMS.exists(), f'Missing best checkpoint: {BEST_PARAMS}'

cmd = [
    sys.executable, '/content/PaddleOCR/tools/eval.py',
    '-c', str(BASE_CONFIG),
    '-o',
    f'Global.checkpoints={BEST_BASE}',
    f'Eval.dataset.data_dir={DATASET_ROOT}',
    f'Eval.dataset.label_file_list=["{test_label_file}"]',
    'Eval.loader.batch_size_per_card=1',
]
if os.system('nvidia-smi > /dev/null 2>&1') != 0:
    cmd.append('Global.use_gpu=False')
run_live(cmd, env=env)

## Backup Artifacts To Drive

In [ ]:
assert OUTPUT_DIR.exists(), f'Missing output dir: {OUTPUT_DIR}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIR, FINAL_MODEL_DRIVE_DIR)
print('Saved detection artifacts to:', FINAL_MODEL_DRIVE_DIR)